# 01.4 - Stacks & Queues

## Stack - LIFO (Last In, First Out)
Like a stack of plates: you add and remove **only at the top**.
```
push(3)      pop() -> 3
 |  3 |       |    |
 |  2 |       |  2 |
 |  1 |       |  1 |
 +----+       +----+
```
## Queue - FIFO (First In, First Out)
Like a line at the bank: join at the **back**, leave from the **front**.
```
enqueue ->  [4][3][2][1]  -> dequeue (1)
```
## Deque - double-ended queue
Add/remove at **both** ends in O(1). Python: `collections.deque`.

## Where are they used?
| Structure | Real system |
|---|---|
| Stack | **Function call stack** (every recursive call!), **undo/redo**, browser **back** button, expression evaluation in compilers/calculators, matching HTML tags, DFS |
| Queue | **Printer queue**, **CPU task scheduling**, web server request queues, **message queues (Kafka, RabbitMQ, Celery)**, BFS, customer support tickets, streaming buffers |
| Deque | Sliding-window max (stock dashboards), work-stealing schedulers, palindrome checks, browser history with limited size |

## Complexity (with the right Python type)
| Operation | Stack (`list`) | Queue (`deque`) |
|---|---|---|
| push / enqueue | O(1) `append` | O(1) `append` |
| pop / dequeue | O(1) `pop()` | O(1) `popleft()` |
| peek | O(1) `s[-1]` | O(1) `q[0]` |

Never use `list.pop(0)` as a queue - it's O(n).

## 1. Build them from scratch

In [1]:
class Stack:
    def __init__(self):
        self._items = []
    def push(self, x):
        self._items.append(x)
    def pop(self):
        if self.is_empty():
            raise IndexError("pop from empty stack")
        return self._items.pop()
    def peek(self):
        return self._items[-1]
    def is_empty(self):
        return not self._items
    def __len__(self):
        return len(self._items)

class Queue:                     # linked-list based: O(1) enqueue & dequeue
    class _Node:
        def __init__(self, val):
            self.val, self.next = val, None
    def __init__(self):
        self.head = self.tail = None
        self.size = 0
    def enqueue(self, x):
        node = self._Node(x)
        if self.tail:
            self.tail.next = node
        else:
            self.head = node
        self.tail = node
        self.size += 1
    def dequeue(self):
        if not self.head:
            raise IndexError("dequeue from empty queue")
        val = self.head.val
        self.head = self.head.next
        if not self.head:
            self.tail = None
        self.size -= 1
        return val
    def __len__(self):
        return self.size

s = Stack()
for x in [1, 2, 3]:
    s.push(x)
print("stack pop order:", [s.pop() for _ in range(3)])   # 3 2 1

q = Queue()
for x in [1, 2, 3]:
    q.enqueue(x)
print("queue dequeue order:", [q.dequeue() for _ in range(3)])  # 1 2 3

stack pop order: [3, 2, 1]
queue dequeue order: [1, 2, 3]


In [2]:
# The Pythonic way
from collections import deque

stack = []
stack.append("a"); stack.append("b")
print(stack.pop(), stack)

queue = deque()
queue.append("job1"); queue.append("job2")
print(queue.popleft(), queue)

# Real-world demo: undo / redo with two stacks
class Editor:
    def __init__(self):
        self.text, self.undo_stack, self.redo_stack = "", [], []
    def type(self, s):
        self.undo_stack.append(self.text)
        self.redo_stack.clear()
        self.text += s
    def undo(self):
        if self.undo_stack:
            self.redo_stack.append(self.text)
            self.text = self.undo_stack.pop()
    def redo(self):
        if self.redo_stack:
            self.undo_stack.append(self.text)
            self.text = self.redo_stack.pop()

e = Editor()
e.type("Hello"); e.type(" World")
e.undo(); print(repr(e.text))
e.redo(); print(repr(e.text))

b ['a']
job1 deque(['job2'])
'Hello'
'Hello World'


---
## Solved problems

### 1. Implement Queue using Stacks - LC #232 (Easy)
Build a FIFO queue using only two stacks; amortised O(1) per operation.

**Idea:** Push into `inbox`. To pop, if `outbox` is empty pour everything from `inbox` into it (this reverses the order) and pop from `outbox`.

In [3]:
class MyQueue:
    def __init__(self):
        self.inbox, self.outbox = [], []
    def push(self, x):
        self.inbox.append(x)
    def _move(self):
        if not self.outbox:                  # only refill when empty
            while self.inbox:
                self.outbox.append(self.inbox.pop())
    def pop(self):
        self._move()
        return self.outbox.pop()
    def peek(self):
        self._move()
        return self.outbox[-1]
    def empty(self):
        return not self.inbox and not self.outbox

q = MyQueue()
q.push(1); q.push(2)
assert q.peek() == 1 and q.pop() == 1 and not q.empty()
q.push(3)
assert q.pop() == 2 and q.pop() == 3 and q.empty()
print("Queue via Stacks ")

Queue via Stacks 


### 2. Evaluate Reverse Polish Notation - LC #150 (Medium)
Evaluate `["2","1","+","3","*"]` → (2+1)×3 = 9. This is how calculators and compilers evaluate expressions.

**Idea:** Numbers go on the stack; an operator pops two numbers, applies, pushes the result.

In [4]:
def eval_rpn(tokens):
    stack = []
    for t in tokens:
        if t in "+-*/":
            b, a = stack.pop(), stack.pop()
            if t == "+": stack.append(a + b)
            elif t == "-": stack.append(a - b)
            elif t == "*": stack.append(a * b)
            else: stack.append(int(a / b))      # truncate toward zero
        else:
            stack.append(int(t))
    return stack[0]

assert eval_rpn(["2", "1", "+", "3", "*"]) == 9
assert eval_rpn(["4", "13", "5", "/", "+"]) == 6
assert eval_rpn(["10", "6", "9", "3", "+", "-11", "*", "/", "*", "17", "+", "5", "+"]) == 22
print("Eval RPN ")

Eval RPN 


### 3. Next Greater Element I - LC #496 (Easy) - monotonic stack
For each `x` in `nums1` (subset of `nums2`), find the first greater element to its right in `nums2`, or -1.

**Idea:** Keep a decreasing stack; when a bigger number arrives, it is the 'next greater' for everything smaller it pops.

In [5]:
def next_greater_element(nums1, nums2):
    next_greater = {}
    stack = []                         # decreasing stack
    for x in nums2:
        while stack and stack[-1] < x:
            next_greater[stack.pop()] = x
        stack.append(x)
    return [next_greater.get(x, -1) for x in nums1]

assert next_greater_element([4, 1, 2], [1, 3, 4, 2]) == [-1, 3, -1]
assert next_greater_element([2, 4], [1, 2, 3, 4]) == [3, -1]
print("Next Greater Element ")

Next Greater Element 


### 4. Design Circular Queue - LC #622 (Medium)
Fixed-size queue that reuses space (ring buffer) - used in audio/video streaming, network drivers, logging.

**Idea:** Array + `front` index + `count`; positions wrap around with `% k`.

In [6]:
class MyCircularQueue:
    def __init__(self, k):
        self.data = [0] * k
        self.k = k
        self.front = 0
        self.count = 0
    def enQueue(self, value):
        if self.isFull():
            return False
        self.data[(self.front + self.count) % self.k] = value
        self.count += 1
        return True
    def deQueue(self):
        if self.isEmpty():
            return False
        self.front = (self.front + 1) % self.k
        self.count -= 1
        return True
    def Front(self):
        return -1 if self.isEmpty() else self.data[self.front]
    def Rear(self):
        return -1 if self.isEmpty() else self.data[(self.front + self.count - 1) % self.k]
    def isEmpty(self):
        return self.count == 0
    def isFull(self):
        return self.count == self.k

cq = MyCircularQueue(3)
assert cq.enQueue(1) and cq.enQueue(2) and cq.enQueue(3) and not cq.enQueue(4)
assert cq.Rear() == 3 and cq.isFull() and cq.deQueue() and cq.enQueue(4) and cq.Rear() == 4 and cq.Front() == 2
print("Circular Queue ")

Circular Queue 


### 5. Sliding Window Maximum - LC #239 (Hard) - monotonic deque Max of every window of size `k`. Real use: rolling max price on stock charts, monitoring dashboards.

**Idea:** Deque of indices with **decreasing** values. The front is always the max of the current window.

In [7]:
from collections import deque

def max_sliding_window(nums, k):
    dq = deque()          # indices; their values are decreasing
    result = []
    for i, x in enumerate(nums):
        while dq and nums[dq[-1]] < x:      # smaller values can never be max again
            dq.pop()
        dq.append(i)
        if dq[0] <= i - k:                  # front left the window
            dq.popleft()
        if i >= k - 1:
            result.append(nums[dq[0]])
    return result

# Time O(n): each index enters and leaves the deque once
assert max_sliding_window([1, 3, -1, -3, 5, 3, 6, 7], 3) == [3, 3, 5, 5, 6, 7]
assert max_sliding_window([1], 1) == [1]
print("Sliding Window Max ")

Sliding Window Max 


## More practice on LeetCode
Easy: Implement Stack using Queues (#225), Baseball Game (#682), Backspace String Compare (#844)
Medium: Decode String (#394), Asteroid Collision (#735), Online Stock Span (#901), Car Fleet (#853), Simplify Path (#71)
Hard: Largest Rectangle in Histogram (#84), Basic Calculator (#224)

Also in `03-Interview Patterns/03.1`: Valid Parentheses, Min Stack, Daily Temperatures.

## Key takeaways
- Stack = `list` (`append`/`pop`); Queue = `deque` (`append`/`popleft`).
- "Matching", "undo", "most recent" → stack. "In order of arrival", BFS → queue.
- "Next greater/smaller" → **monotonic stack**. "Window max/min" → **monotonic deque**.

**Next: `01.5-Trees-and-BST.ipynb`**